## Importación de paquetes

In [42]:
import os
import numpy as np                  # Importación de la biblioteca numpy para el manejo de arreglos
import pandas as pd                 # Importación de la biblioteca pandas para el manejo de datos contenidos en una base de datos
import matplotlib.pyplot as plt     # Importación de la biblioteca para realizar la visualización de resultados mediante elementos gráficos
import seaborn as sns               # Importación de la biblioteca para realizar la visualización de resultados mediante elementos gráficos
np.set_printoptions(
    suppress=True,
    linewidth=100,
    precision=2
)
plt.rcParams['figure.dpi'] = 120

## Funciones Auxiliares

### Gráfico de Barra

Función utilizada para la **generación de gráficos de barras**. 

Para su correcta ejecución, es necesario proporcionar los siguientes parámetros:

- `data_pandas`: conjunto de datos a representar gráficamente, almacenado en una estructura de tipo DataFrame de la librería pandas.

- `cols`: arreglo que contiene los nombres de las columnas que se desean graficar, definidos exactamente como aparecen en el DataFrame.

- `titulo`: cadena de caracteres utilizada para establecer el título del gráfico.

- `xlabel`: cadena de caracteres empleada para identificar la variable representada en el eje x.

- `ylabel`: cadena de caracteres empleada para identificar la variable representada en el eje y.

In [43]:
def plot_grafico_barra(data_pandas, cols, titulo, xlabel, ylabel, ancho=7, alto=5):
    
    sns.set_theme(
        style="ticks",
        context="notebook",
        font_scale=1.0
    )

    # Dimensiones
    FIG_WIDTH = ancho
    FIG_HEIGHT = alto

    # Selección de columnas
    df =data_pandas.loc[:, cols]

    # Figura
    fig, ax = plt.subplots(
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        dpi=120
    )

    # Gráfico
    sns.barplot(
        data=df,
        x=cols[0],
        y=cols[1],
        palette='tab10',
        ax=ax,
        color='black'
    )

    # Título
    ax.set_title(
        titulo,
        fontsize=16,
        fontweight='bold',
        pad=15
    )

    # Etiquetas
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    # Rotación de etiquetas
    ax.tick_params(axis='x', rotation=45)

    # Valores sobre barras
    for container in ax.containers:
        ax.bar_label(
            container,
            fontsize=10,
            padding=3
        )

    # Cuadrícula elegante
    ax.grid(axis='y', linestyle='-', alpha=0.8)

    # Bordes limpios
    sns.despine()

    plt.tight_layout()
    plt.show()

## Obtención de la Ruta del Notebook

In [44]:
os.getcwd()

'd:\\Análisis de Datos\\test_prestamo_bancario\\Project 01 - Data Cleaning and Preprocessing'

## Importación de los Datos

In [45]:
nombre_archivo_datos = 'loan-data.csv'     # Nombre del archivo que contiene los datos a procesar

Durante esta etapa se llevan a cabo las siguientes actividades:

- Extracción de la información contenida en las bases de datos proporcionadas en formato numpy.array.

- Identificación y clasificación del tipo de dato presente en cada una de las variables que conforman el conjunto de datos.

- Segmentación de la base de datos original en dos grupos principales: 
    * datos categóricos y/o binarios
    * datos numéricos

Esta segmentación se realiza con el propósito de aplicar posteriormente los procedimientos de limpieza y preprocesamiento adecuados para cada tipo de información.

### **Extracción de los Datos Proporcionados**

La carga de la información se realiza considerando todos los registros como cadenas de texto. La carga se realiza de esta manera para evitar la ocurrencia de errores durante el proceso de lectura y almacenamiento de los datos. 

Se emplea el punto y coma (`;`) como delimitador para separar los distintos campos contenidos en el conjunto de datos. La selección de este delimitador se definió a partir de analizar el formato de los datos crudos proporcionados.

In [ ]:
datos_crudos_np = np.loadtxt(nombre_archivo_datos, delimiter=';', dtype='str')     # Carga de los datos proporcionados en formato de cadena de caracteres para realizar, a posteriori, la clasificación de su formato
datos_crudos_np = datos_crudos_np.transpose()                                      # Conversión de columnas a filas para un mejor procesamiento de los datos para su clasificación y casteo
etiquetas = datos_crudos_np[:,0]                                                   # Extracción de las etiquetas (primera fila de la base de datos)
datos = datos_crudos_np[:,1:]                                                      # Extracción de los datos (filas restantes de la base de datos a partir de la segunda)
print(etiquetas)
print(datos[:,:5])

['id' 'issue_d' 'loan_amnt' 'loan_status' 'funded_amnt' 'term' 'int_rate' 'installment' 'grade'
 'sub_grade' 'verification_status' 'url' 'addr_state' 'total_pymnt']
[['48010226' '57693261' '59432726' '53222800' '57803010']
 ['May-15' '' 'Sep-15' 'Jul-15' 'Aug-15']
 ['35000.0' '30000.0' '15000.0' '9600.0' '8075.0']
 ['Current' 'Current' 'Current' 'Current' 'Current']
 ['35000.0' '30000.0' '15000.0' '9600.0' '8075.0']
 [' 36 months' ' 36 months' ' 36 months' ' 36 months' ' 36 months']
 ['13.33' 'þëè.89' 'íîå.53' 'þëè.89' '19.19']
 ['1184.86' '938.57' '494.86' '300.35' '296.78']
 ['C' 'A' 'B' 'A' '']
 ['C3' 'A5' 'B5' 'A5' 'E3']
 ['Verified' 'Source Verified' 'Verified' 'Not Verified' 'Source Verified']
 ['https://www.lendingclub.com/browse/loanDetail.action?loan_id=48010226'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=57693261'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=59432726'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=53222

Como resultado del proceso de carga y extracción de información, se generan las siguientes estructuras de datos:

- `etiquetas`: arreglo que contiene los nombres de las variables o características presentes en el conjunto de datos original. Del análisis de los datos crudos proporcionados se identifica que estas etiquetas se encuentran registradas en la primera fila de la base de datos. Su función es describir cada atributo y facilitar la identificación e interpretación de la información almacenada.

- `datos`: estructura que almacena los valores correspondientes a cada uno de los registros del conjunto de datos, manteniendo la asociación entre los datos y las etiquetas que los describen. Esta información está compuesta por todas las filas comprendidas entre la segunda y la última fila de la base de datos proporcionada.

### **Formato de los Datos Extraídos**

Una vez cargados los registros, se realiza un proceso de formateo de los datos con el objetivo de convertir cada elemento a un valor numérico de tipo punto flotante. 

Cuando un registro no puede convertirse debido a 

- incompatibilidades en su formato 

- presencia de valores faltantes

Se le asigna el valor especial `np.nan`, el cual permite identificar y gestionar posteriormente los datos no válidos o ausentes durante el análisis.

In [47]:
datos_numericos = []
for i in datos:
    temporal = []
    for j in i:
        try:
            x = float(j)
        except:
            x = np.nan
        finally:
            temporal.append(x)
    datos_numericos.append(temporal)
datos_numericos = np.array(datos_numericos)
print(datos_numericos[:,:5])

[[48010226.   57693261.   59432726.   53222800.   57803010.  ]
 [        nan         nan         nan         nan         nan]
 [   35000.      30000.      15000.       9600.       8075.  ]
 [        nan         nan         nan         nan         nan]
 [   35000.      30000.      15000.       9600.       8075.  ]
 [        nan         nan         nan         nan         nan]
 [      13.33         nan         nan         nan       19.19]
 [    1184.86      938.57      494.86      300.35      296.78]
 [        nan         nan         nan         nan         nan]
 [        nan         nan         nan         nan         nan]
 [        nan         nan         nan         nan         nan]
 [        nan         nan         nan         nan         nan]
 [        nan         nan         nan         nan         nan]
 [    9452.96     4679.7      1969.83     1793.68     1178.51]]


Como resultado de este proceso se obtienen la siguiente variable:

- `datos_numericos`: conjunto de datos cuyos elementos han sido convertidos a valores numéricos de tipo punto flotante. Aquellos registros que no pudieron ser transformados debido a incompatibilidades en su formato se representan mediante el valor especial `np.nan`.

### **Clasificación de las Agrupaciones de Datos**

Se realiza un análisis de las variables que componen la base de datos con el propósito de identificar aquellas cuyos valores pueden representarse mediante datos numéricos. Para ello, se calcula el porcentaje de valores nulos resultantes respecto al total de registros disponibles.

A partir de este procedimiento se establecen los siguientes criterios:

* Si el porcentaje de valores nulos es igual al 100 %, se concluye que ninguno de los registros de la categoría pudo ser interpretado como un valor numérico, por lo que la variable se considera de naturaleza no numérica.
* Si el porcentaje de valores nulos es inferior al 100 %, se determina que la categoría contiene al menos algunos valores susceptibles de ser representados numéricamente.
* Si el porcentaje de valores nulos es mayor que 0 %, la categoría presenta registros que no pudieron convertirse a formato numérico o que originalmente correspondían a datos faltantes. En estos casos, podría ser necesario aplicar técnicas de limpieza e imputación durante las etapas posteriores del análisis.

Cabe destacar que la capacidad de una variable para representarse mediante valores numéricos no implica necesariamente que esta corresponda a una variable cuantitativa. En muchos conjuntos de datos existen categorías, etiquetas o identificadores que se encuentran codificados mediante números, aun cuando dichos valores no poseen un significado numérico desde el punto de vista analítico.

Por esta razón, la clasificación definitiva de una variable requiere analizar no solo el formato de los datos que contiene, sino también su propósito dentro del conjunto de datos y la naturaleza de la información que representa.

In [48]:
cantidad_nulos = []
for i in datos_numericos:
    cantidad_nulos.append(int(pd.isna(i).sum()/len(i)*100))
indice_categoricos, indice_numericos = [], []
for i in range(len(cantidad_nulos)):
    if cantidad_nulos[i] == 100:
        indice_categoricos.append(i)
    else:
        indice_numericos.append(i)
print(indice_numericos)
print(indice_categoricos)

[0, 2, 4, 6, 7, 13]
[1, 3, 5, 8, 9, 10, 11, 12]


### **Cambio de Nombre de las Etiquetas**

In [51]:
dic_etiquetas = {
    'id' : 'identificador',
    'issue_d' : 'fecha_prestamo',
    'loan_status' : 'estado_prestamo',
    'loan_amnt' : 'solicitud_prestamo',
    'funded_amnt' : 'aceptacion_prestamo',
    'term' : 'meses_prestamo',
    'grade' : 'grado',
    'sub_grade' : 'sub_grado',
    'verification_status' : 'estado_verificacion',
    'int_rate' : 'razon_intereses',
    'installment' : 'pago_mensual',
    'total_pymnt' : 'pago_efectuado',
    'url' : 'enlace',
    'addr_state' : 'estado',
}

### **Visualización de los Datos Crudos**

In [66]:
datos_pandas = pd.DataFrame(datos.transpose(), columns=etiquetas)
datos_pandas.rename(columns=dic_etiquetas, inplace=True)
datos_pandas.head()

,identificador,fecha_prestamo,solicitud_prestamo,estado_prestamo,aceptacion_prestamo,meses_prestamo,razon_intereses,pago_mensual,grado,sub_grado,estado_verificacion,enlace,estado,pago_efectuado
0,48010226,May-15,35000.0,Current,35000.0,36 months,13.33,1184.86,C,C3,Verified,https://www.lendingclub.com/browse/loanDetail....,CA,9452.96
1,57693261,,30000.0,Current,30000.0,36 months,þëè.89,938.57,A,A5,Source Verified,https://www.lendingclub.com/browse/loanDetail....,NY,4679.7
2,59432726,Sep-15,15000.0,Current,15000.0,36 months,íîå.53,494.86,B,B5,Verified,https://www.lendingclub.com/browse/loanDetail....,PA,1969.83
3,53222800,Jul-15,9600.0,Current,9600.0,36 months,þëè.89,300.35,A,A5,Not Verified,https://www.lendingclub.com/browse/loanDetail....,OH,1793.68
4,57803010,Aug-15,8075.0,Current,8075.0,36 months,19.19,296.78,,E3,Source Verified,https://www.lendingclub.com/browse/loanDetail....,TX,1178.51


In [53]:
etiquetas_previas = etiquetas.copy()
etiquetas = []
for i in etiquetas_previas:
    etiquetas.append(dic_etiquetas[i])
print(etiquetas)

['identificador', 'fecha_prestamo', 'solicitud_prestamo', 'estado_prestamo', 'aceptacion_prestamo', 'meses_prestamo', 'razon_intereses', 'pago_mensual', 'grado', 'sub_grado', 'estado_verificacion', 'enlace', 'estado', 'pago_efectuado']


In [54]:
print('Variables no numéricas')
for i in indice_categoricos:
    print(etiquetas[i])

Variables no numéricas
fecha_prestamo
estado_prestamo
meses_prestamo
grado
sub_grado
estado_verificacion
enlace
estado


In [56]:
print('Variables numéricas')
columnas_numericas = []
for i in indice_numericos:
    columnas_numericas.append(etiquetas[i])
    print(etiquetas[i])

Variables numéricas
identificador
solicitud_prestamo
aceptacion_prestamo
razon_intereses
pago_mensual
pago_efectuado


### **Cambio de Formato a Numérico**

La variable  `id` se encuentra representada mediante valores numéricos; sin embargo, su función dentro del conjunto de datos es actuar como identificador único de cada préstamo registrado. Por esta razón, aunque sus valores puedan interpretarse como datos numéricos desde el punto de vista del formato de almacenamiento, no constituyen una variable cuantitativa sobre la cual resulte apropiado realizar análisis estadísticos o aplicar operaciones aritméticas.

En consecuencia, esta variable se clasifica como un identificador y se excluye del conjunto de variables numéricas consideradas para el análisis. Alternativamente, puede integrarse dentro del grupo de variables categóricas o de identificación, preservando su utilidad para la trazabilidad de los registros.

In [67]:
datos_pandas[columnas_numericas[1:]] = datos_pandas[columnas_numericas[1:]].apply(pd.to_numeric, errors = 'coerce')

In [68]:
datos_pandas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   identificador        10000 non-null  object 
 1   fecha_prestamo       10000 non-null  object 
 2   solicitud_prestamo   9500 non-null   float64
 3   estado_prestamo      10000 non-null  object 
 4   aceptacion_prestamo  9500 non-null   float64
 5   meses_prestamo       10000 non-null  object 
 6   razon_intereses      3996 non-null   float64
 7   pago_mensual         9499 non-null   float64
 8   grado                10000 non-null  object 
 9   sub_grado            10000 non-null  object 
 10  estado_verificacion  10000 non-null  object 
 11  enlace               10000 non-null  object 
 12  estado               10000 non-null  object 
 13  pago_efectuado       9500 non-null   float64
dtypes: float64(5), object(9)
memory usage: 1.1+ MB


### **Observaciones por Variables**

Existen diversas relaciones que pueden establecerse entre los valores contenidos en las variables que conforman la base de datos proporcionada:

* Se analiza la posible correlación entre las variables `loan_amnt` y `funded_amnt`, con el objetivo de evaluar la viabilidad de un proceso de imputación de valores ausentes en ambas columnas, en caso de existir una relación estadísticamente significativa entre ellas.

* Se identifica una relación directa entre la codificación de las variables `sub_grade` y `grade`, lo que permite considerar la posibilidad de realizar procesos de imputación cruzada en aquellos casos en los que se presenten valores faltantes en alguna de estas variables, aprovechando su correspondencia estructural.

### **Generación CSV CheckPoint**

In [71]:
datos_pandas.to_csv('data_v1.csv')